In [1]:
import pandas as pd

df = pd.read_csv("shopping_table (1).csv")

print(df.shape)
print(df.head())

(3900, 6)
   Age  Purchase_Amount_USD  Review_Rating  Previous_Purchases  \
0   55                   53            3.1                  14   
1   19                   64            3.1                   2   
2   50                   73            3.1                  23   
3   21                   90            3.5                  49   
4   45                   49            2.7                  31   

  Discount_Applied Subscription_Status  
0              Yes                 Yes  
1              Yes                 Yes  
2              Yes                 Yes  
3              Yes                 Yes  
4              Yes                 Yes  


In [2]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  3900 non-null   int64  
 1   Purchase_Amount_USD  3900 non-null   int64  
 2   Review_Rating        3900 non-null   float64
 3   Previous_Purchases   3900 non-null   int64  
 4   Discount_Applied     3900 non-null   object 
 5   Subscription_Status  3900 non-null   object 
dtypes: float64(1), int64(3), object(2)
memory usage: 182.9+ KB
None

Missing values:
Age                    0
Purchase_Amount_USD    0
Review_Rating          0
Previous_Purchases     0
Discount_Applied       0
Subscription_Status    0
dtype: int64


In [3]:
from sklearn.preprocessing import StandardScaler

# Convert Yes/No values to 1/0
df["Discount_Applied"] = df["Discount_Applied"].map({"Yes": 1, "No": 0})
df["Subscription_Status"] = df["Subscription_Status"].map({"Yes": 1, "No": 0})

# Select the features for clustering
features = [
    "Age",
    "Purchase_Amount_USD",
    "Review_Rating",
    "Previous_Purchases",
    "Discount_Applied",
    "Subscription_Status"
]

X = df[features]

print(X.head())
print("\nMissing values:")
print(X.isnull().sum())

C:\Users\roopa\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


   Age  Purchase_Amount_USD  Review_Rating  Previous_Purchases  \
0   55                   53            3.1                  14   
1   19                   64            3.1                   2   
2   50                   73            3.1                  23   
3   21                   90            3.5                  49   
4   45                   49            2.7                  31   

   Discount_Applied  Subscription_Status  
0                 1                    1  
1                 1                    1  
2                 1                    1  
3                 1                    1  
4                 1                    1  

Missing values:
Age                    0
Purchase_Amount_USD    0
Review_Rating          0
Previous_Purchases     0
Discount_Applied       0
Subscription_Status    0
dtype: int64


In [4]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(3900, 6)


In [5]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

print(df["Cluster"].value_counts().sort_index())

0    1417
1    1430
2    1053
Name: Cluster, dtype: int64


In [6]:
cluster_summary = df.groupby("Cluster")[[
    "Age",
    "Purchase_Amount_USD",
    "Review_Rating",
    "Previous_Purchases",
    "Discount_Applied",
    "Subscription_Status"
]].mean().round(2)

print(cluster_summary)

           Age  Purchase_Amount_USD  Review_Rating  Previous_Purchases  \
Cluster                                                                  
0        44.10                39.23           3.68               24.18   
1        43.92                80.31           3.82               25.97   
2        44.23                59.49           3.74               26.08   

         Discount_Applied  Subscription_Status  
Cluster                                         
0                    0.23                  0.0  
1                    0.21                  0.0  
2                    1.00                  1.0  


In [7]:
print("Cluster Sizes:")
print(df["Cluster"].value_counts().sort_index())

print("\nCluster Summary:")
print(cluster_summary)

Cluster Sizes:
0    1417
1    1430
2    1053
Name: Cluster, dtype: int64

Cluster Summary:
           Age  Purchase_Amount_USD  Review_Rating  Previous_Purchases  \
Cluster                                                                  
0        44.10                39.23           3.68               24.18   
1        43.92                80.31           3.82               25.97   
2        44.23                59.49           3.74               26.08   

         Discount_Applied  Subscription_Status  
Cluster                                         
0                    0.23                  0.0  
1                    0.21                  0.0  
2                    1.00                  1.0  


In [8]:
profile = df.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    Avg_Age=("Age", "mean"),
    Avg_Spend=("Purchase_Amount_USD", "mean"),
    Avg_Rating=("Review_Rating", "mean"),
    Avg_Previous_Purchases=("Previous_Purchases", "mean"),
    Discount_Rate=("Discount_Applied", "mean"),
    Subscription_Rate=("Subscription_Status", "mean")
).round(2)

print(profile)

         Customers  Avg_Age  Avg_Spend  Avg_Rating  Avg_Previous_Purchases  \
Cluster                                                                      
0             1417    44.10      39.23        3.68                   24.18   
1             1430    43.92      80.31        3.82                   25.97   
2             1053    44.23      59.49        3.74                   26.08   

         Discount_Rate  Subscription_Rate  
Cluster                                    
0                 0.23                0.0  
1                 0.21                0.0  
2                 1.00                1.0  


In [9]:
df.to_csv("customer_segmentation_clusters.csv", index=False)

print("Clustering results saved successfully.")

Clustering results saved successfully.


In [10]:
print(profile.to_string())

         Customers  Avg_Age  Avg_Spend  Avg_Rating  Avg_Previous_Purchases  Discount_Rate  Subscription_Rate
Cluster                                                                                                     
0             1417    44.10      39.23        3.68                   24.18           0.23                0.0
1             1430    43.92      80.31        3.82                   25.97           0.21                0.0
2             1053    44.23      59.49        3.74                   26.08           1.00                1.0


In [11]:
summary = df.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    Avg_Age=("Age", "mean"),
    Avg_Spend=("Purchase_Amount_USD", "mean"),
    Avg_Rating=("Review_Rating", "mean"),
    Avg_Previous_Purchases=("Previous_Purchases", "mean"),
    Discount_Rate=("Discount_Applied", "mean"),
    Subscription_Rate=("Subscription_Status", "mean")
).round(2)

summary["Discount_Rate"] = summary["Discount_Rate"] * 100
summary["Subscription_Rate"] = summary["Subscription_Rate"] * 100

summary

,Customers,Avg_Age,Avg_Spend,Avg_Rating,Avg_Previous_Purchases,Discount_Rate,Subscription_Rate
Cluster,,,,,,,
0,1417,44.10,39.23,3.68,24.18,23.0,0.0
1,1430,43.92,80.31,3.82,25.97,21.0,0.0
2,1053,44.23,59.49,3.74,26.08,100.0,100.0


In [12]:
print(summary)

         Customers  Avg_Age  Avg_Spend  Avg_Rating  Avg_Previous_Purchases  \
Cluster                                                                      
0             1417    44.10      39.23        3.68                   24.18   
1             1430    43.92      80.31        3.82                   25.97   
2             1053    44.23      59.49        3.74                   26.08   

         Discount_Rate  Subscription_Rate  
Cluster                                    
0                 23.0                0.0  
1                 21.0                0.0  
2                100.0              100.0  


In [13]:
cluster_names = {
    0: "Budget / Low-Spending Customers",
    1: "High-Spending Customers",
    2: "Subscribed & Discount-Oriented Customers"
}

df["Customer_Segment"] = df["Cluster"].map(cluster_names)

df.to_csv("customer_segmentation_final.csv", index=False)

print(df[["Cluster", "Customer_Segment"]].drop_duplicates().sort_values("Cluster"))

      Cluster                          Customer_Segment
1055        0           Budget / Low-Spending Customers
1053        1                   High-Spending Customers
0           2  Subscribed & Discount-Oriented Customers
